In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025/MC-7_D-2_To Baunia.xlsx
/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025/IC-1_D-2_Kalshi.xlsx
/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025/MC-7_D-1_To Jashimuddin.xlsx
/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025/MC-5_D-1_To Jashimuddin.xlsx
/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025/MC-9_D-1_To Housebuilding.xlsx
/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025/IC-1_D-1_Dhaka Cantonment.xlsx
/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025/MC-9_D-2_To Diabari.xlsx
/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025/MC-1_D-2_To Airport.xlsx
/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025/MC-2_D-1_To Jashimuddin.xlsx
/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025/MC-3_D-2_To Mirpur 12.xlsx
/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025/MC-1_D-1_To Kuril.xlsx
/kaggle/input/master-od-matrix-analysis-datase

## Step 1 — Import libraries + set paths

In [2]:
import os
import glob
import numpy as np
import pandas as pd
from openpyxl import load_workbook

DATASET_DIR = "/kaggle/input/master-od-matrix-analysis-dataset-21-dec-2025"
MASTER_PATH = os.path.join(DATASET_DIR, "Master Excel.xlsx")
OUT_PATH = "/kaggle/working/Master Excel_Aggregated.xlsx"

VEH_SHEETS = [f"vehicle_type-{i}" for i in range(1, 26)]
MATRIX_SIZE = 63


## Step 2 — List ONLY location files (exclude Master)

In [3]:
all_xlsx = glob.glob(os.path.join(DATASET_DIR, "*.xlsx"))

location_files = []
for f in all_xlsx:
    base = os.path.basename(f)
    if base.lower().startswith("~$"):
        continue
    if os.path.abspath(f) == os.path.abspath(MASTER_PATH):
        continue
    location_files.append(f)

print("Location files found:", len(location_files))
for f in location_files:
    print(" -", os.path.basename(f))


Location files found: 19
 - MC-7_D-2_To Baunia.xlsx
 - IC-1_D-2_Kalshi.xlsx
 - MC-7_D-1_To Jashimuddin.xlsx
 - MC-5_D-1_To Jashimuddin.xlsx
 - MC-9_D-1_To Housebuilding.xlsx
 - IC-1_D-1_Dhaka Cantonment.xlsx
 - MC-9_D-2_To Diabari.xlsx
 - MC-1_D-2_To Airport.xlsx
 - MC-2_D-1_To Jashimuddin.xlsx
 - MC-3_D-2_To Mirpur 12.xlsx
 - MC-1_D-1_To Kuril.xlsx
 - MC-10_D-1_To Metro Station.xlsx
 - IC-1_D-3_Balughat.xlsx
 - MC-10_D-2_To Jashimuddin.xlsx
 - MC-5_D-2_To ECB.xlsx
 - MC-8_D-2_To ECB.xlsx
 - MC-3_D-1_To Diabari.xlsx
 - MC-8_D-1_To Kalshi.xlsx
 - MC-2_D-2_To Baunia Bazar.xlsx


## Step 3 — Helper: find the “O/D” anchor cell in a sheet

In [4]:
def find_matrix_anchor(df_raw: pd.DataFrame):
    s = df_raw.astype(str)

    mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
    coords = np.argwhere(mask.values)

    if coords.size == 0:
        raise ValueError("Could not find 'O/D' anchor cell in this sheet.")

    r, c = coords[0]
    return int(r), int(c)


## Step 4 — Helper: extract the 63×63 OD matrix from one sheet

In [5]:
def extract_od_matrix(xlsx_path: str, sheet_name: str) -> np.ndarray:
    df_raw = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None)

    r0, c0 = find_matrix_anchor(df_raw)

    # matrix starts one row below & one column right of 'O/D'
    r_start = r0 + 1
    c_start = c0 + 1

    block = df_raw.iloc[r_start:r_start + MATRIX_SIZE, c_start:c_start + MATRIX_SIZE]

    # Convert to numeric; NaN -> 0
    mat = pd.to_numeric(block.stack(), errors="coerce").unstack(fill_value=0).values.astype(float)

    if mat.shape != (MATRIX_SIZE, MATRIX_SIZE):
        raise ValueError(f"Matrix shape {mat.shape}, expected {(MATRIX_SIZE, MATRIX_SIZE)}")

    return mat


## Step 5 — Test extraction on one file, one sheet

In [6]:
test_file = location_files[0]
test_sheet = "vehicle_type-1"

mat = extract_od_matrix(test_file, test_sheet)

print("Test file:", os.path.basename(test_file))
print("Sheet:", test_sheet)
print("Shape:", mat.shape)
print("Total trips in this matrix:", mat.sum())


Test file: MC-7_D-2_To Baunia.xlsx
Sheet: vehicle_type-1
Shape: (63, 63)
Total trips in this matrix: 18.0


/tmp/ipykernel_55/2471061319.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])


## Step 6 — Aggregate all 19 files for all 25 vehicle-type sheets

In [7]:
agg = {sh: np.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=float) for sh in VEH_SHEETS}

for fpath in location_files:
    for sh in VEH_SHEETS:
        agg[sh] += extract_od_matrix(fpath, sh)

print("Aggregation complete.")
print("Example total (vehicle_type-1):", agg["vehicle_type-1"].sum())


/tmp/ipykernel_55/2471061319.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
/tmp/ipykernel_55/2471061319.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
/tmp/ipykernel_55/2471061319.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
/tmp/ipykernel_55/2471061319.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
/tmp/ipykernel_55/2471061319.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d",

Aggregation complete.
Example total (vehicle_type-1): 523.0


/tmp/ipykernel_55/2471061319.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])


## Step 7 — Helper: write aggregated matrix into Master (sheet-wise)

In [8]:
def write_matrix_to_sheet(ws, mat: np.ndarray):
    # Find 'O/D' anchor in master sheet
    anchor = None
    for row in ws.iter_rows():
        for cell in row:
            if cell.value is None: 
                continue
            v = str(cell.value).strip().lower()
            if v in ["o/d", "o\\d", "od", "o / d"]: 
                anchor = (cell.row, cell.column)  # openpyxl is 1-based
                break
        if anchor:
            break

    if not anchor:
        raise ValueError(f"Could not find 'O/D' in master sheet '{ws.title}'")

    r0, c0 = anchor
    r_start = r0 + 1
    c_start = c0 + 1

    for i in range(MATRIX_SIZE):
        for j in range(MATRIX_SIZE):
            ws.cell(row=r_start + i, column=c_start + j).value = float(mat[i, j])


## Step 8 — Write all vehicle sheets into Master and save output

In [9]:
wb = load_workbook(MASTER_PATH)

for sh in VEH_SHEETS:
    ws = wb[sh]
    write_matrix_to_sheet(ws, agg[sh])

wb.save(OUT_PATH)
print("Saved aggregated master to:", OUT_PATH)


Saved aggregated master to: /kaggle/working/Master Excel_Aggregated.xlsx


## Step 9 — Quick validation (optional but recommended)

In [10]:
check = pd.read_excel(OUT_PATH, sheet_name="vehicle_type-1", header=None)
print("Read-back successful. Open the output file to confirm.") 

Read-back successful. Open the output file to confirm.


# ONE SINGLE MASTER FILE

In [42]:
import os
import glob
import numpy as np
import pandas as pd
from openpyxl import load_workbook

MASTER_AGG_PATH = "/kaggle/working/Master Excel_Aggregated.xlsx"
OUT_TOTAL_PATH  = "/kaggle/working/OD_Total_All_Vehicles.xlsx"

VEH_SHEETS = [f"vehicle_type-{i}" for i in range(1, 26)]
MATRIX_SIZE = 63


In [44]:
def find_matrix_anchor(df_raw: pd.DataFrame):
    s = df_raw.astype(str)
    mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
    coords = np.argwhere(mask.values)
    if coords.size == 0:
        raise ValueError("Could not find 'O/D' anchor cell.")
    r, c = coords[0]
    return int(r), int(c)


In [45]:
def extract_od_matrix(xlsx_path: str, sheet_name: str) -> np.ndarray:
    df_raw = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None)

    r0, c0 = find_matrix_anchor(df_raw)
    r_start = r0 + 1
    c_start = c0 + 1

    block = df_raw.iloc[r_start:r_start + MATRIX_SIZE, c_start:c_start + MATRIX_SIZE]
    mat = pd.to_numeric(block.stack(), errors="coerce").unstack(fill_value=0).values.astype(float)

    if mat.shape != (MATRIX_SIZE, MATRIX_SIZE):
        raise ValueError(f"Matrix shape {mat.shape}, expected {(MATRIX_SIZE, MATRIX_SIZE)}")

    return mat


In [46]:
total_mat = np.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=float)

for sh in VEH_SHEETS:
    total_mat += extract_od_matrix(MASTER_AGG_PATH, sh)

print("Total matrix created.")
print("Grand total trips (sum of all cells):", total_mat.sum())


/tmp/ipykernel_47/544124148.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
/tmp/ipykernel_47/544124148.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
/tmp/ipykernel_47/544124148.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
/tmp/ipykernel_47/544124148.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
/tmp/ipykernel_47/544124148.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\

Total matrix created.
Grand total trips (sum of all cells): 23135.0


/tmp/ipykernel_47/544124148.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
/tmp/ipykernel_47/544124148.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])


In [47]:
total_df = pd.DataFrame(total_mat)

with pd.ExcelWriter(OUT_TOTAL_PATH, engine="openpyxl") as writer:
    total_df.to_excel(writer, sheet_name="OD_Total_All_Vehicles", index=False, header=False)

print("Saved:", OUT_TOTAL_PATH) 

Saved: /kaggle/working/OD_Total_All_Vehicles.xlsx


# Freight Wise OD Matrix Master File

In [15]:
import numpy as np
import pandas as pd

# -------------------------
# Paths (Kaggle)
# -------------------------
MASTER_AGG_PATH = "/kaggle/working/Master Excel_Aggregated.xlsx"
#OUT_PATH = "/kaggle/working/Truck_OD_Matrix_VehicleTypes_20-24.xlsx"
#OUT_PATH_Bus = "/kaggle/working/BUS_OD_Matrix_VehicleTypes_14-19.xlsx"
#OUT_PATH_Car = "/kaggle/working/Car_OD_Matrix_VehicleTypes_9-12.xlsx"
#OUT_PATH_NMT = "/kaggle/working/NMT_OD_Matrix_VehicleTypes_1-5.xlsx"
OUT_PATH_CTA = "/kaggle/working/CNG,Tempo,Auto_OD_Matrix_VT_7,8,13.xlsx"

# -------------------------
# Select only these sheets
# -------------------------
#SELECTED_SHEETS = [f"vehicle_type-{i}" for i in range(1, 6)]  # 20,21,22,23,24
SELECTED_SHEETS = [
    "vehicle_type-7",
    "vehicle_type-8",
    "vehicle_type-13"
]
MATRIX_SIZE = 63

# -------------------------
# Helpers
# -------------------------
def find_matrix_anchor(df_raw: pd.DataFrame):
    s = df_raw.astype(str)
    mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
    coords = np.argwhere(mask.values)
    if coords.size == 0:
        raise ValueError("Could not find 'O/D' anchor cell.")
    r, c = coords[0]
    return int(r), int(c)

def extract_od_matrix(xlsx_path: str, sheet_name: str) -> np.ndarray:
    df_raw = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None)
    r0, c0 = find_matrix_anchor(df_raw)

    # Matrix starts one row below and one column right of 'O/D'
    r_start = r0 + 1
    c_start = c0 + 1

    block = df_raw.iloc[r_start:r_start + MATRIX_SIZE, c_start:c_start + MATRIX_SIZE]
    mat = pd.to_numeric(block.stack(), errors="coerce").unstack(fill_value=0).values.astype(float)

    if mat.shape != (MATRIX_SIZE, MATRIX_SIZE):
        raise ValueError(f"Matrix shape {mat.shape}, expected {(MATRIX_SIZE, MATRIX_SIZE)}")

    return mat

# -------------------------
# Sum selected vehicle types
# -------------------------
total_mat = np.zeros((MATRIX_SIZE, MATRIX_SIZE), dtype=float)

for sh in SELECTED_SHEETS:
    total_mat += extract_od_matrix(MASTER_AGG_PATH, sh)

print("Selected sheets summed:", SELECTED_SHEETS)
print("Grand total (sum of all OD cells):", total_mat.sum())

# -------------------------
# Save to new Excel (one sheet only)
# -------------------------
total_df = pd.DataFrame(total_mat)

with pd.ExcelWriter(OUT_PATH_CTA, engine="openpyxl") as writer:
    total_df.to_excel(writer, sheet_name="CNG,Tempo,Auto_OD_Matrix_VT_7,8,13", index=False, header=False)

print("Saved:", OUT_PATH_CTA)

/tmp/ipykernel_55/3911978767.py:30: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
/tmp/ipykernel_55/3911978767.py:30: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])


Selected sheets summed: ['vehicle_type-7', 'vehicle_type-8', 'vehicle_type-13']
Grand total (sum of all OD cells): 6192.0
Saved: /kaggle/working/CNG,Tempo,Auto_OD_Matrix_VT_7,8,13.xlsx


/tmp/ipykernel_55/3911978767.py:30: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  mask = s.applymap(lambda x: str(x).strip().lower() in ["o/d", "o\\d", "od", "o / d"])
/usr/local/lib/python3.12/dist-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
